
# Workflow Practice

In this notebook, you’ll practice connecting to a SQLite database, creating tables from CSV files using Pandas, and writing SQL queries to explore the data.

The dataset comes from the [Bike Store Sample Database](https://www.kaggle.com/datasets/dillonmyrick/bike-store-sample-database) by Dillon Myrick. It models a fictional bike retailer with multiple stores, products, customers, and staff. Each table connects to others using foreign keys such as `customer_id`, `store_id`, and `product_id`.

You’ll:
- Connect to a local SQLite database
- Create tables using `pandas.to_sql()`
- Write and test SQL queries using `pd.read_sql()`

All of your work will take place directly in this notebook. Each question prompt is written below as a Markdown cell, followed by an empty code cell for you to write your query.



## Step 1: Connect to the Database

Run the following cell to connect to (or create) a SQLite database called `bike_store.db`.  
If the file doesn’t exist yet, SQLite will automatically create it.


In [22]:
import sqlite3
import pandas as pd
from pathlib import Path

In [23]:
connection = sqlite3.connect("bike_store.db")
connection


## Step 2: Create Tables from CSV Files

The `data/` folder contains one CSV file per table.  
Use `pandas.read_csv()` and `DataFrame.to_sql()` to load each file into your database.

You only need to do this once.  
After that, you’ll be able to run queries against your newly created tables.


In [24]:
# Example for one file
customers = pd.read_csv("data/customers.csv")
customers.to_sql("customers", connection, if_exists="replace", index=False)

1445

In [25]:
# Repeat for all other files in the data folder, or use a loop.
data_folder = Path("data/")

for file_path in data_folder.iterdir():
    df = pd.read_csv(file_path)
    table_name = file_path.stem
    df.to_sql(
        table_name,
        connection,
        if_exists="replace",
        index=False
    )
    print(f"Loaded {file_path.name} to {table_name}")


Loaded brands.csv to brands
Loaded categories.csv to categories
Loaded customers.csv to customers
Loaded orders.csv to orders
Loaded order_items.csv to order_items
Loaded products.csv to products
Loaded staffs.csv to staffs
Loaded stocks.csv to stocks
Loaded stores.csv to stores


### Verify Your Tables

Run a query to make sure your tables were created successfully.

In [26]:

pd.read_sql("SELECT name FROM sqlite_master WHERE type='table';", connection)


,name
0,brand_id brand_name\n0 1 E...
1,category_id category_name\n0 ...
2,customer_id first_name last_name ...
3,order_id customer_id order_status ord...
4,order_id item_id product_id quantity ...
5,product_id produ...
6,staff_id first_name last_name ...
7,store_id product_id quantity\n0 ...
8,store_id store_name phone ...
9,brands


## Step 3: Test a Simple Query

Before starting the exercises, confirm your connection and tables are working by previewing the first few rows of the `customers` table.

In [27]:

pd.read_sql("SELECT discount from order_items", connection)


,discount
0,0.20
1,0.07
2,0.05
3,0.05
4,0.20
...,...
4717,0.07
4718,0.20
4719,0.20
4720,0.07


### Q1. List all customers and their cities.

Return the first name, last name, and city of each customer. Sort alphabetically by last name and then by first name.

In [28]:
# Your query here
pd.read_sql("""
SELECT first_name, last_name, city
FROM customers
ORDER BY last_name, first_name;
""", connection)

,first_name,last_name,city
0,Ester,Acevedo,San Lorenzo
1,Jamika,Acevedo,Ozone Park
2,Penny,Acevedo,Ballston Spa
3,Bettyann,Acosta,Lancaster
4,Shery,Acosta,Saratoga Springs
...,...,...,...
1440,Edda,Young,North Tonawanda
1441,Jasmin,Young,Helotes
1442,Alexandria,Zamora,Schenectady
1443,Jayme,Zamora,Springfield Gardens


### Q2. Show all products and their prices.

Display each product name along with its list price. Sort by price in descending order.

In [29]:
# Your query here
query2 = """ 
SELECT product_name, list_price
FROM products
ORDER BY list_price DESC


"""
pd.read_sql(query2, connection)

,product_name,list_price
0,Trek Domane SLR 9 Disc - 2018,11999.99
1,Trek Domane SLR 8 Disc - 2018,7499.99
2,Trek Silque SLR 8 Women's - 2017,6499.99
3,Trek Domane SL Frameset - 2018,6499.99
4,Trek Domane SL Frameset Women's - 2018,6499.99
...,...,...
316,Trek Kickster - 2018,159.99
317,Trek Boy's Kickster - 2015/2017,149.99
318,Trek Girl's Kickster - 2017,149.99
319,Sun Bicycles Lil Kitt'n - 2017,109.99


### Q3. Find all customers from California.

Return first name, last name, city, and state for all customers whose state is 'CA'. Sort alphabetically by last name.

In [35]:
# Your query here
query3 = """ 
SELECT first_name, last_name, city, state
FROM customers
WHERE state = 'CA'
ORDER BY last_name ASC
"""
pd.read_sql(query3, connection)

,first_name,last_name,city,state
0,Ester,Acevedo,San Lorenzo,CA
1,Jamaal,Albert,Torrance,CA
2,Sindy,Anderson,Pomona,CA
3,Twana,Arnold,Anaheim,CA
4,Selene,Austin,Duarte,CA
...,...,...,...,...
279,Darren,Witt,Coachella,CA
280,Lucy,Woods,Palos Verdes Peninsula,CA
281,Joel,Wynn,San Diego,CA
282,Yvone,Yates,San Pablo,CA


### Q4. Count how many products are in each category.

Return the category name and the number of products in that category. Sort from the highest count to the lowest.

In [39]:
# Your query here
query4 = """ 
SELECT count(p.product_name) as number_products, c.category_name
FROM products p
JOIN categories c
ON p.category_id = c.category_id
GROUP BY c.category_name
ORDER BY number_products DESC
"""
pd.read_sql(query4, connection)

,number_products,category_name
0,78,Cruisers Bicycles
1,60,Road Bikes
2,60,Mountain Bikes
3,59,Children Bicycles
4,30,Comfort Bicycles
5,24,Electric Bikes
6,10,Cyclocross Bicycles


### Q5. Find all orders placed in 2018.

List the order ID, order date, and customer ID for orders made during the year 2018. Sort by order date.

In [40]:
# Your query here
query5 = """ 

SELECT order_id, order_date, customer_id
FROM orders
WHERE order_date LIKE '2018%'
ORDER BY order_date

"""
pd.read_sql(query5, connection)

,order_id,order_date,customer_id
0,1324,2018-01-01,862
1,1325,2018-01-01,68
2,1326,2018-01-01,567
3,1327,2018-01-02,1026
4,1328,2018-01-02,1083
...,...,...,...
287,1611,2018-09-06,6
288,1612,2018-10-21,3
289,1613,2018-11-18,1
290,1614,2018-11-28,135


### Q6. Show each order with its total number of items.

Join the `orders` and `order_items` tables. Group by order ID and return the number of items per order.

In [42]:
# Your query here
query6 = """

SELECT oi.order_id, sum(oi.quantity) as number_items
FROM order_items oi
JOIN orders o
ON oi.order_id = o.order_id
GROUP BY o.order_id

"""
pd.read_sql(query6, connection)

,order_id,number_items
0,1,8
1,2,3
2,3,2
3,4,2
4,5,4
...,...,...
1610,1611,4
1611,1612,8
1612,1613,3
1613,1614,5


### Q7. List total revenue per store.

Revenue = quantity * list_price * (1 - discount). Join `orders`, `order_items`, and `stores`, group by store name, and return total revenue.

In [50]:
# Your query here
query7 = """ 

SELECT s.store_name as store_name, sum(oi.quantity * oi.list_price *(1 - oi.discount)) as total_revenue
FROM stores s
JOIN orders o 
    ON o.store_id = s.store_id
JOIN order_items oi 
    ON oi.order_id = o.order_id
GROUP BY store_name

"""
df = pd.read_sql(query7, connection)

# Originally returned scientific notation for revenue number
df['total_revenue'] = df['total_revenue'].round(2)

# Display dataframe with updated revenue 
df

,store_name,total_revenue
0,Baldwin Bikes,5215751.28
1,Rowlett Bikes,867542.24
2,Santa Cruz Bikes,1605823.04


### Q8. Find the top 5 customers who spent the most overall.

Join `customers`, `orders`, and `order_items`. Sum the total spending per customer and return the top five spenders.

In [55]:
# Your query here
query8 = """ 

SELECT concat(c.first_name," ",c.last_name) as customer_name, sum(oi.quantity * oi.list_price *(1 - oi.discount)) as total_spending
FROM customers c
JOIN orders o 
    ON c.customer_id = o.customer_id
JOIN order_items oi 
    ON o.order_id = oi.order_id
GROUP BY customer_name
ORDER BY total_spending desc
LIMIT 5


"""
pd.read_sql(query8, connection)

,customer_name,total_spending
0,Sharyn Hopkins,34807.9392
1,Pamelia Newman,33634.2604
2,Abby Gamble,32803.0062
3,Lyndsey Bean,32675.0725
4,Emmitt Sanchez,31925.8857


### Q9. Show the best-selling product in each category.

Join `products`, `order_items`, and `categories`. For each category, identify the product with the highest total quantity sold.

In [60]:
# Your query here

# Window function to select max per category 
query9 = """ 
WITH product_sales AS ( 
    SELECT 
        p.product_name as product_name,
        c.category_name as category_name,
        sum(oi.quantity) as total_quantity
    FROM products p 
    JOIN categories c 
        ON p.category_id = c.category_id
    JOIN order_items oi
        ON p.product_id = oi.product_id
    GROUP BY p.product_name, c.category_name
)
SELECT
    ps.product_name,
    ps.category_name,
    ps.total_quantity
FROM 
    product_sales ps
JOIN (
    SELECT 
        category_name,
        max(total_quantity) as max_quantity
    FROM product_sales
    GROUP BY category_name
) max_sales 
    ON 
        ps.category_name = max_sales.category_name
        AND 
        ps.total_quantity = max_sales.max_quantity
GROUP BY ps.category_name;


"""
pd.read_sql(query9, connection)

,product_name,category_name,total_quantity
0,Electra Girl's Hawaii 1 (20-inch) - 2015/2016,Children Bicycles,154
1,Electra Townie Original 7D - 2015/2016,Comfort Bicycles,148
2,Electra Cruiser 1 (24-Inch) - 2016,Cruisers Bicycles,157
3,Surly Straggler 650b - 2016,Cyclocross Bicycles,151
4,Trek Conduit+ - 2016,Electric Bikes,145
5,Surly Ice Cream Truck Frameset - 2016,Mountain Bikes,167
6,Trek Domane SLR 6 Disc - 2017,Road Bikes,43


### Q10. Identify the employees (staff) who processed the most orders.

Join `staffs` and `orders`. Count the number of orders handled by each staff member and return the results sorted by highest total.

In [61]:
# Your query here
query10 = """ 

SELECT 
    concat(s.first_name, " ", s.last_name) as staff_member, 
    count(o.order_id) as orders_handled
FROM staffs s
JOIN orders o
    ON o.staff_id = s.staff_id
GROUP BY staff_member 
ORDER BY orders_handled DESC




"""
pd.read_sql(query10, connection)

,staff_member,orders_handled
0,Marcelene Boyer,553
1,Venita Daniel,540
2,Genna Serrano,184
3,Mireya Copeland,164
4,Kali Vargas,88
5,Layla Terrell,86
